### Week 6, Day 2

Before proceeding with our own MCP Server, let's just look at 2 popular marketplaces for what's out there:

https://glama.ai/mcp  
https://smithery.ai/servers 

We're about to create and use our own MCP Server!

It's pretty simple, but it's not super-simple. The excitement around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

backend/accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

## Any guesses where this Account python module came from?!

I didn't write it!

In [3]:
from backend.accounts import Account

In [8]:
account = Account.get("Siddharth")
account.reset()
account

Account(name='siddharth', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [9]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "siddharth", "balance": 9106.64686, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 297.78438, "timestamp": "2026-09-19 04:49:38", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-09-19 04:49:38", 9998.21686]], "total_portfolio_value": 9998.21686, "total_profit_loss": -1.7831399999995483}'

In [10]:
account.report()

'{"name": "siddharth", "balance": 9106.64686, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 297.78438, "timestamp": "2026-09-19 04:49:38", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-09-19 04:49:38", 9998.21686], ["2026-09-19 04:50:40", 9998.03686]], "total_portfolio_value": 9998.03686, "total_profit_loss": -1.9631399999998393}'

In [11]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 297.78438,
  'timestamp': '2026-09-19 04:49:38',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [15]:
# Now let's use our accounts server as an MCP server

Params = {"command":"uv", "args": ["run", "-m", "backend.accounts_server"]}
async with MCPServerStdio(params=Params, client_session_timeout_seconds=30) as Server:
    MCPTools = await Server.list_tools()

MCPTools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder    \n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='get_holdings', title=None, description='Get the holdings od the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execut

In [17]:
Instructions = "You are able to manage an account for a client, and answer questions about the account"
Request = "My name is Siddharth and my account is under the name Siddharth. What is my balance and my holdings?"
Model="gpt-5.4-mini"

In [18]:
async with MCPServerStdio(params=Params, client_session_timeout_seconds=30) as Server:
    agent = Agent(name="AccountManager", instructions=Instructions, model=model, mcp_servers=[Server])
    Result = await Runner.run(agent,Request)
    display(Markdown(Result.final_output))

Your account balance is **$9,106.65**.

Your holdings are:
- **AMZN**: 3 shares

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to send a push notification, and then
            enjoy the outcome! There is a solution in backend/push_server.py if you need a hint.
            </span>
        </td>
    </tr>
</table>